In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import MinMaxScaler

In [2]:
orders = pd.read_csv("olist_orders_dataset_clean.csv")
items = pd.read_csv("olist_order_items_dataset_clean.csv")
payments = pd.read_csv("olist_order_payments_dataset_clean.csv")
reviews = pd.read_csv("olist_order_reviews_dataset_clean.csv")
customers = pd.read_csv("olist_customers_dataset_clean.csv")

In [3]:
df = orders.merge(reviews, on="order_id", how="left")
df = df.merge(customers, on="customer_id", how="left")

In [4]:
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

In [5]:
item_count = items.groupby('order_id').size().reset_index(name='raw_item_count')
df = df.merge(item_count, on='order_id', how='left')

In [6]:
order_value = items.groupby('order_id')['price'].sum().reset_index()
order_value.rename(columns={'price':'raw_order_value'}, inplace=True)

df = df.merge(order_value, on='order_id', how='left')

In [7]:
freight_total = items.groupby('order_id')['freight_value'].sum().reset_index()
freight_total.rename(columns={'freight_value':'raw_freight_total'}, inplace=True)

df = df.merge(freight_total, on='order_id', how='left')

In [8]:
df['raw_freight_ratio'] = df['raw_freight_total'] / df['raw_order_value']

In [9]:
df['raw_delivery_time'] = (
    df['order_delivered_customer_date'] -
    df['order_purchase_timestamp']
).dt.days

In [10]:
df['raw_estimated_delivery'] = (
    df['order_estimated_delivery_date'] -
    df['order_purchase_timestamp']
).dt.days

In [11]:
df['raw_delivery_delay'] = (
    df['order_delivered_customer_date'] -
    df['order_estimated_delivery_date']
).dt.days

In [12]:
df['raw_seller_processing_time'] = (
    df['order_delivered_carrier_date'] -
    df['order_purchase_timestamp']
).dt.days

In [13]:
customer_orders = orders.groupby('customer_id').size().reset_index()

customer_orders.rename(columns={0:'raw_customer_orders'}, inplace=True)

df = df.merge(customer_orders, on='customer_id', how='left')

In [14]:
raw_cols = [c for c in df.columns if c.startswith('raw_')]

df[raw_cols] = df[raw_cols].fillna(df[raw_cols].median())

In [15]:
for col in raw_cols:
    
    q1 = df[col].quantile(0.01)
    q3 = df[col].quantile(0.99)

    df[col] = np.clip(df[col], q1, q3)

In [16]:
scaler = MinMaxScaler()

norm_cols = [c.replace('raw_', 'norm_') for c in raw_cols]

df[norm_cols] = scaler.fit_transform(df[raw_cols])

In [17]:
df[norm_cols]

,norm_item_count,norm_order_value,norm_freight_total,norm_freight_ratio,norm_delivery_time,norm_estimated_delivery,norm_delivery_delay,norm_seller_processing_time,norm_customer_orders
0,0.0,0.018259,0.013663,0.185964,0.159091,0.204545,0.518519,0.117647,0.0
1,0.0,0.108244,0.157900,0.116652,0.272727,0.295455,0.555556,0.058824,0.0
2,0.0,0.150036,0.121533,0.066573,0.181818,0.454545,0.333333,0.000000,0.0
3,0.0,0.033485,0.203513,0.405535,0.272727,0.454545,0.425926,0.176471,0.0
4,0.0,0.008024,0.013663,0.289161,0.022727,0.136364,0.481481,0.000000,0.0
...,...,...,...,...,...,...,...,...,...
99987,0.0,0.060873,0.058455,0.109598,0.159091,0.272727,0.462963,0.058824,0.0
99988,0.0,0.165252,0.130573,0.062879,0.477273,0.386364,0.629630,0.058824,0.0
99989,0.0,0.196789,0.592048,0.203382,0.522727,0.545455,0.555556,0.058824,0.0
99990,0.5,0.352993,0.758065,0.140290,0.363636,0.704545,0.277778,0.176471,0.0


In [18]:
df['target'] = df['review_score'].apply(
    lambda x: 0 if x <= 2 else (1 if x == 3 else 2)
)

In [19]:
features = norm_cols

X = df[features]

y = df['target']

In [20]:
print(y.value_counts(normalize=True))

target
2    0.772442
0    0.145762
1    0.081797
Name: proportion, dtype: float64


In [21]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

X_res, y_res = smote.fit_resample(X, y)

In [22]:
from collections import Counter
print(Counter(y_res))

Counter({2: 77238, 0: 77238, 1: 77238})


In [23]:
import xgboost as xgb

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

In [24]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [25]:
model = xgb.XGBClassifier(
    objective='multi:softprob',
    num_class=3,
    eval_metric='mlogloss',
    max_depth=6,
    learning_rate=0.1,
    n_estimators=200,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

In [26]:
model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=200, n_jobs=None, num_class=3, ...)

In [27]:
y_pred = model.predict(X_test)

In [28]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.70      0.40      0.51      2915
           1       0.00      0.00      0.00      1636
           2       0.82      0.98      0.89     15448

    accuracy                           0.81     19999
   macro avg       0.51      0.46      0.47     19999
weighted avg       0.74      0.81      0.76     19999



In [29]:
df['target'] = df['review_score'].apply(lambda x: 1 if x >= 4 else 0)

In [30]:
features = norm_cols

X = df[features]
y = df['target']

In [31]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [32]:
import xgboost as xgb

model = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    max_depth=6,
    learning_rate=0.1,
    n_estimators=200,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=200, n_jobs=None,
              num_parallel_tree=None, ...)

In [33]:
y_pred = model.predict(X_test)


In [34]:
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.78      0.32      0.46      4705
           1       0.82      0.97      0.89     15294

    accuracy                           0.82     19999
   macro avg       0.80      0.65      0.68     19999
weighted avg       0.81      0.82      0.79     19999



In [35]:
df=pd.read_csv("c:/Users/Ngoc Mai/OneDrive/Documents/Olist/olist_full_merged.csv")

In [36]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 113425 entries, 0 to 113424
Data columns (total 36 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   order_id                       113425 non-null  object 
 1   customer_id                    113425 non-null  object 
 2   order_status                   113425 non-null  object 
 3   order_purchase_timestamp       113425 non-null  object 
 4   order_approved_at              113425 non-null  object 
 5   order_delivered_carrier_date   113425 non-null  object 
 6   order_delivered_customer_date  113425 non-null  object 
 7   order_estimated_delivery_date  113425 non-null  object 
 8   customer_unique_id             113425 non-null  object 
 9   customer_zip_code_prefix       113425 non-null  int64  
 10  customer_city                  113425 non-null  object 
 11  customer_state                 113425 non-null  object 
 12  order_item_id                 

In [37]:
df[df['customer_unique_id'] == '0093c499f4f9c0e169ead490aa62ce3e'][['customer_id','order_id','review_score']]

,customer_id,order_id,review_score
22488,7f6d1f1be2482d7a6c1d0083eed8f6fb,34f275d9e157cbaa33ca327cf52e19b0,5.0


In [42]:
import pandas as pd

df_llm = pd.read_csv(r"c:/Users/Ngoc Mai/Downloads/llm_predictions (2).csv")

df_llm.head()

,customer_unique_id,P_LLM
0,0004aac84e0df4da2b147fca70cf8255,0.999995
1,00053a61a98854899e70ed204dd4bafe,0.999995
2,0000f46a3911fa3c0805444483337064,0.999995
3,00090324bbad0e9342388303bb71ba0a,0.999995
4,000ec5bff359e1c0ad76a81a45cb598f,0.586495


In [65]:
df["true_label"] = (df["review_score"] <= 4).astype(int)

In [40]:
df_customer = df.groupby("customer_unique_id")["true_label"].mean().reset_index()

df_customer["true_label"] = (df_customer["true_label"] > 0).astype(int)

In [43]:
df_eval = df_llm.merge(
    df_customer,
    on="customer_unique_id",
    how="inner"
)

df_eval.head()

,customer_unique_id,P_LLM,true_label
0,0004aac84e0df4da2b147fca70cf8255,0.999995,0
1,00053a61a98854899e70ed204dd4bafe,0.999995,1
2,0000f46a3911fa3c0805444483337064,0.999995,1
3,00090324bbad0e9342388303bb71ba0a,0.999995,0
4,000ec5bff359e1c0ad76a81a45cb598f,0.586495,0


In [45]:
df_eval["P_LLM"].unique()[:20]

array([0.999995, 0.586495, 0.999994, 0.85    , 1.      , 0.385791,
       0.45    , 0.786   , 0.291666, 0.8     , 0.25    , 0.75    ,
       0.657934, 0.995   , 0.579999, 0.3     , 0.2     , 0.666667,
       0.283485, 0.834995])

In [71]:
df_eval["pred_label"] = (df_eval["P_LLM"] < 0.879).astype(int)

In [72]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(df_eval["true_label"], df_eval["pred_label"])

print("Accuracy:", accuracy)

Accuracy: 0.5903076677825602


In [73]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(df_eval["true_label"], df_eval["pred_label"])
print(cm)

[[2968  616]
 [1954  735]]


In [69]:
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np
import pandas as pd

def find_best_threshold(df, prob_col="P_LLM", true_col="true_label"):

    results = []

    thresholds = np.arange(0.1, 0.9, 0.01)

    for t in thresholds:

        pred = (df[prob_col] < t).astype(int)

        precision = precision_score(df[true_col], pred)
        recall = recall_score(df[true_col], pred)
        f1 = f1_score(df[true_col], pred)

        results.append({
            "threshold": t,
            "precision": precision,
            "recall": recall,
            "f1": f1
        })

    result_df = pd.DataFrame(results)

    best = result_df.loc[result_df["f1"].idxmax()]

    print("Best threshold:", best["threshold"])
    print("Precision:", best["precision"])
    print("Recall:", best["recall"])
    print("F1:", best["f1"])

    return result_df, best

In [70]:
result_df, best = find_best_threshold(df_eval)

Best threshold: 0.8799999999999996
Precision: 0.5440414507772021
Recall: 0.2733358125697285
F1: 0.36386138613861385
